## Exploration des données

In [73]:
import pandas as pd

In [74]:
df_raw = pd.read_csv("data/all_data_with_identities.csv")
df_raw.head()

,Unnamed: 0,id,comment_text,split,created_date,publication_id,parent_id,article_id,rating,funny,...,asian_latino_etc,disability_any,identity_any,num_identities,more_than_one_identity,na_gender,na_orientation,na_religion,na_race,na_disability
0,0,627762,OH yes - Were those evil Christian Missionarie...,test,2016-11-26 15:56:03.862109+00,13,627198.0,152737,approved,0,...,0,0,1,1.0,False,1,1,0,1,1
1,1,5892815,Why is this black racist crap still on the G&M...,val,2017-09-03 23:20:08.226613+00,54,NaN,373428,rejected,0,...,0,0,1,2.0,True,1,1,1,0,1
2,2,416437,even up here.......BLACKS!,train,2016-08-04 16:48:07.175252+00,21,NaN,143025,rejected,0,...,0,0,1,1.0,False,1,1,1,0,1
3,3,5137126,Blame men. There's always an excuse to blame ...,train,2017-04-15 19:00:45.032674+00,54,5136907.0,327125,rejected,0,...,0,0,1,2.0,True,0,1,1,1,1
4,4,855753,And the woman exposing herself saying grab thi...,val,2017-01-18 01:50:57.478867+00,13,849081.0,162008,rejected,0,...,0,0,1,1.0,False,0,1,1,1,1


In [75]:
df_raw.columns

Index(['Unnamed: 0', 'id', 'comment_text', 'split', 'created_date',
       'publication_id', 'parent_id', 'article_id', 'rating', 'funny', 'wow',
       'sad', 'likes', 'disagree', 'toxicity', 'severe_toxicity', 'obscene',
       'sexual_explicit', 'identity_attack', 'insult', 'threat', 'male',
       'female', 'transgender', 'other_gender', 'heterosexual',
       'homosexual_gay_or_lesbian', 'bisexual', 'other_sexual_orientation',
       'christian', 'jewish', 'muslim', 'hindu', 'buddhist', 'atheist',
       'other_religion', 'black', 'white', 'asian', 'latino',
       'other_race_or_ethnicity', 'physical_disability',
       'intellectual_or_learning_disability', 'psychiatric_or_mental_illness',
       'other_disability', 'identity_annotator_count',
       'toxicity_annotator_count', 'LGBTQ', 'other_religions',
       'asian_latino_etc', 'disability_any', 'identity_any', 'num_identities',
       'more_than_one_identity', 'na_gender', 'na_orientation', 'na_religion',
       'na_race', 

In [76]:
columns = ['comment_text', 'split', 'toxicity', 'male', 'female', 'transgender', 'other_gender', 'black', 'white', 'asian', 'latino', 'other_race_or_ethnicity', 'heterosexual', 'homosexual_gay_or_lesbian', 'bisexual', 'other_sexual_orientation']
df = df_raw[columns]

In [61]:
text_lengths = df.comment_text.astype('str').apply(len)
text_lengths.describe()

count    448000.000000
mean        351.742958
std         289.323717
min           1.000000
25%         121.000000
50%         257.000000
75%         512.000000
max        1891.000000
Name: comment_text, dtype: float64

In [62]:
text_lengths[df.toxicity > 0.5].describe()

count    36000.000000
mean       296.576194
std        251.508949
min          3.000000
25%        111.000000
50%        213.000000
75%        399.000000
max       1000.000000
Name: comment_text, dtype: float64

In [63]:
text_lengths[df.toxicity < 0.5].describe()

count    397206.000000
mean        355.287999
std         292.107289
min           1.000000
25%         121.000000
50%         260.000000
75%         521.000000
max        1891.000000
Name: comment_text, dtype: float64

## 3. Classification

On cherche à entraîner un modèle de classification permettant l'identification d'un texte toxique en fonction de l'occurence des mots évoquant le genre, l'ethnie ou l'orientation sexuelle dans ce texte. 

Puisqu'il s'agit d'une tâche de classification, on binarise la toxicité avec un seuil à 0.5 :

In [82]:
df_clean = df.drop(columns=["comment_text"])
df_clean.toxicity = df_clean.toxicity >= 0.5

La base de données possède déjà une colonne "split", que l'on peut utiliser pour constuire les datasets d'entraînement, de test et de validation.

In [83]:
df_train = df_clean[df_clean.split=="train"].drop(columns=["split"])
df_test = df_clean[df_clean.split=="test"].drop(columns=["split"])
df_val = df_clean[df_clean.split=="val"].drop(columns=["split"])

df_train.head()

,toxicity,male,female,transgender,other_gender,black,white,asian,latino,other_race_or_ethnicity,heterosexual,homosexual_gay_or_lesbian,bisexual,other_sexual_orientation
2,True,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,True,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,True,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,True,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,True,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Puisqu'on ne s'intéresse pas au texte complet, mais plutôt à la présence ou non de mots évoquant le genre, l'éthnie ou l'orientation sexuelle représentés par les autres colonnes de la base de données, on peut se permettre l'utilisation d'un modèle léger pour la classification. La toxicité étant représentée initialement comme un nombre compris entre 0 et 1, une régression logistique peut apparaître comme un choix de modèle pertinent. 

### Logistic Regression

In [97]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np

In [111]:
X_train, X_test = df_train.drop(columns=["toxicity"]), df_test.drop(columns=["toxicity"])
y_train, y_test = df_train.toxicity, df_test.toxicity

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
    ))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:,1]
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

       False       0.91      0.82      0.86    118558
        True       0.21      0.38      0.27     15224

    accuracy                           0.77    133782
   macro avg       0.56      0.60      0.57    133782
weighted avg       0.83      0.77      0.80    133782

ROC AUC: 0.6311450660049743


Malgré la prise en compte de poids pour équilibrer le fait que la proportion des commentaires toxique est inférieure à 10%, le modèle donne de très mauvais résultats pour identifier les commentaires toxiques. Avec un recall de 38% seulement, la majorité des commentaires toxiques ne sont ici pas détectés ! 

In [112]:
y_test_sample = y_test.to_numpy().astype(np.uint8)[:1000]
y_pred_sample = y_pred.astype(np.uint8)[:1000]

## Explicabilité

In [113]:
logreg_model = pipeline.named_steps["model"]

coef = logreg_model.coef_[0]
features = X_train.columns

n=len(y_test_sample)
residuals = y_test_sample - y_pred_sample
residual_sum_of_squares = residuals.T @ residuals
sigma_squared_hat = residual_sum_of_squares / (n - 2)
var_beta = np.array([1/np.sum([(X_test[col].to_numpy()[j] - np.mean(X_test[col]))**2 for j in range(n)]) * sigma_squared_hat for col in features])
SE_beta = np.sqrt(var_beta)
t_beta = coef/SE_beta

print(f"{'Feature':<40} {'Weight':>12} {'SE':>12} {'|t|':>12}")
print("-" * 60)

for col, i in zip(features, range(len(features))):
    print(f"{col:<40} {coef[i]:>12.6f} {SE_beta[i]:>12.6f} {abs(t_beta[i]):>12.6f}")

Feature                                        Weight           SE          |t|
------------------------------------------------------------
male                                         0.060012     0.028380     2.114626
female                                       0.098030     0.024098     4.067946
transgender                                  0.042111     0.113672     0.370461
other_gender                                 0.003180     2.323329     0.001369
black                                        0.207869     0.033428     6.218449
white                                        0.262515     0.023416    11.210922
asian                                       -0.014017     0.119709     0.117092
latino                                       0.014892     0.181877     0.081878
other_race_or_ethnicity                      0.016194     0.233236     0.069431
heterosexual                                -0.002904     0.228234     0.012722
homosexual_gay_or_lesbian                    0.201251     0

On observe que les caractéristiques les plus déterminantes dans la classification des messages toxiques est la présence des mots "blanc" et "noir" dans un contexte ethnique (plus que les mots désignant les autres ethnies), ou la présence des mots "homme" et "femme" (plus que celle des autres genres). On retrouve les caractéristiques avec les taux de toxicité les plus importants.

## Fairness

In [116]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix

identity_columns = [
    'male', 'female', 'transgender', 'other_gender', 
    'black', 'white', 'asian', 'latino', 'other_race_or_ethnicity', 
    'heterosexual', 'homosexual_gay_or_lesbian', 'bisexual', 'other_sexual_orientation'
]

def fairness_evaluation_with_trueDI(y_true, y_pred, X):    
    df_eval = X.copy()
    df_eval['y_true'] = y_true
    df_eval['y_pred'] = y_pred
    
    results = {}
    
    for group_col in identity_columns:
        
        # Binarisation explicite
        df_eval[group_col + "_bin"] = df_eval[group_col] > 0.5
        
        metrics = []
        
        # Définir groupe privilégié (ex: False)
        privileged_group = False
        
        for g in [True, False]:
            sub = df_eval[df_eval[group_col + "_bin"] == g]
            
            if len(sub) == 0:
                continue
            
            # Confusion matrix
            tn, fp, fn, tp = confusion_matrix(
                sub['y_true'], sub['y_pred'], labels=[0,1]
            ).ravel()
            
            fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
            fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
            
            # DI prédiction
            p_priv_pred = df_eval[df_eval[group_col + "_bin"] == privileged_group]['y_pred'].mean()
            p_unpriv_pred = sub['y_pred'].mean()
            di_pred = p_unpriv_pred / p_priv_pred if p_priv_pred > 0 else np.nan
            
            # DI vrai label
            p_priv_true = df_eval[df_eval[group_col + "_bin"] == privileged_group]['y_true'].mean()
            p_unpriv_true = sub['y_true'].mean()
            di_true = p_unpriv_true / p_priv_true if p_priv_true > 0 else np.nan
            
            metrics.append({
                'group': g,
                'count': len(sub),
                'FPR': fpr,
                'FNR': fnr,
                'DI_pred': di_pred,
                'DI_true': di_true
            })
        
        results[group_col] = pd.DataFrame(metrics)
    
    return results

results = fairness_evaluation_with_trueDI(y_test_sample, y_pred_sample, X_test[:1000])

for group_col, df_metrics in results.items():
    print(f"\n{group_col}")
    print(df_metrics.to_string(index=False))


male
 group  count      FPR      FNR  DI_pred  DI_true
  True     96 0.518519 0.449275 1.958667 0.958333
 False    904 0.212389 0.702065 1.000000 1.000000

female
 group  count      FPR      FNR  DI_pred  DI_true
  True    126 0.821429 0.102041 4.031164 1.047423
 False    874 0.173333 0.765794 1.000000 1.000000

transgender
 group  count      FPR      FNR  DI_pred  DI_true
  True      6 1.000000 0.000000 3.358108 1.116352
 False    994 0.242063 0.683288 1.000000 1.000000

other_gender
 group  count      FPR      FNR  DI_pred  DI_true
 False   1000 0.245059 0.678715      1.0      1.0

black
 group  count      FPR      FNR  DI_pred  DI_true
  True     61 1.000000 0.000000 3.896266 1.247515
 False    939 0.229839 0.733719 1.000000 1.000000

white
 group  count      FPR     FNR  DI_pred  DI_true
  True    132 1.000000 0.00000 5.105882 1.075474
 False    868 0.154867 0.78972 1.000000 1.000000

asian
 group  count      FPR      FNR  DI_pred  DI_true
  True      3      NaN 0.333333 2.215556 

On observe qu'il existe un léger biais dans les données réelles : la caractéristique "male" est légèrement sous-représentée dans les commentaires toxiques, avec un score de 0.96, tandis que les autres caractéristiques sont légèrement sur-représentées, avec la caractéristique "black" en tête à 1.25. Ces biais sont très largement amplifiés dans les prédictions, allant d'un facteur 2 pour la caractéristique "male" jusqu'à 4 pour la caractéristique "female" : le biais du modèle est donc massif et a tendance à prédire de nombreux faux positifs, et ce malgré un taux de faux négatifs également conséquent.